# 06 — Analysis Dataset Construction & Sample Audit

**Project:** WR-MovementIQ  
**Purpose:** Build and audit the player-season modelling table from the independently verified Combine and game player-season profiles.

This notebook does not fit models. It establishes the analysis population, joins the verified feature tables, defines the research variables, and exports sample-flow and missingness audits.

**Primary inclusion rule:** WR/SLOT_WR player-seasons at or after the player's draft year, with at least 10 receiver plays.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Resolve project root whether the notebook is launched from the project root
# or from its notebooks/ subdirectory.
_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in [_here, *_here.parents]
     if (candidate / "data" / "features").exists()),
    _here
)
# If automatic detection fails, set PROJECT_ROOT manually.
DATA_FEATURES = PROJECT_ROOT / "data" / "features"
DATA_RAW = PROJECT_ROOT / "data" / "raw"
REPORTS = PROJECT_ROOT / "reports" / "analysis_dataset"
REPORTS.mkdir(parents=True, exist_ok=True)

COMBINE_PROFILE_PATH = DATA_FEATURES / "wr_combine_player_profiles.parquet"
GAME_PROFILE_PATH = DATA_FEATURES / "wr_game_player_season_profiles.parquet"
PLAYERS_PATH = DATA_RAW / "players.csv"

for path in [COMBINE_PROFILE_PATH, GAME_PROFILE_PATH, PLAYERS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required input not found: {path}")

combine = pd.read_parquet(COMBINE_PROFILE_PATH)
game = pd.read_parquet(GAME_PROFILE_PATH)
players = pd.read_csv(PLAYERS_PATH, low_memory=False)

print("INPUT SHAPES")
print("Combine player profiles:", combine.shape)
print("Game player-season profiles:", game.shape)
print("Players:", players.shape)

print("\\nCOMBINE COLUMNS")
print(combine.columns.tolist())
print("\\nGAME PROFILE COLUMNS")
print(game.columns.tolist())
print("\\nPLAYER COLUMNS")
print(players.columns.tolist())

INPUT SHAPES
Combine player profiles: (107, 30)
Game player-season profiles: (146, 41)
Players: (510, 10)
\nCOMBINE COLUMNS
['draft_year', 'nfl_id', 'primary_drills_observed', 'primary_attempts_observed', 'combine_mean_speed_median_across_drills', 'combine_mean_speed_iqr_across_drills', 'combine_peak_speed_median_across_drills', 'combine_peak_speed_iqr_across_drills', 'combine_p95_speed_median_across_drills', 'combine_p95_speed_iqr_across_drills', 'combine_mean_acceleration_median_across_drills', 'combine_mean_acceleration_iqr_across_drills', 'combine_peak_acceleration_median_across_drills', 'combine_peak_acceleration_iqr_across_drills', 'combine_coordinate_path_length_median_across_drills', 'combine_coordinate_path_length_iqr_across_drills', 'combine_net_displacement_median_across_drills', 'combine_net_displacement_iqr_across_drills', 'combine_path_efficiency_median_across_drills', 'combine_path_efficiency_iqr_across_drills', 'combine_coordinate_total_turn_median_across_drills', 'comb

## 1. Validate source tables and keys

The expected grain is one row per `draft_year, nfl_id` in the Combine profile and one row per `draft_year, nfl_id, season` in the game profile. We stop if these grains are not unique.

In [2]:
COMBINE_KEY = ["draft_year", "nfl_id"]
GAME_KEY = ["draft_year", "nfl_id", "season"]

def require_columns(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")

require_columns(combine, COMBINE_KEY, "Combine profile")
require_columns(game, GAME_KEY + ["receiver_plays"], "Game player-season profile")
require_columns(players, ["nfl_id", "nfl_position"], "players.csv")

def key_audit(df, keys, name):
    dup = int(df.duplicated(keys).sum())
    nulls = int(df[keys].isna().any(axis=1).sum())
    print(f"{name}: rows={len(df):,}; duplicate keys={dup:,}; null-key rows={nulls:,}")
    if dup or nulls:
        raise ValueError(f"{name} failed key-grain validation.")

key_audit(combine, COMBINE_KEY, "Combine profile")
key_audit(game, GAME_KEY, "Game player-season profile")

# Keep one player-level position record per NFL ID. Conflicting positions are audited.
position_audit = (
    players.dropna(subset=["nfl_id"])
    .groupby("nfl_id")["nfl_position"]
    .nunique(dropna=True)
)
conflicting_ids = position_audit[position_audit > 1]
print("Player IDs with conflicting nfl_position values:", len(conflicting_ids))
if len(conflicting_ids):
    raise ValueError("Conflicting player positions found; resolve before building the analysis table.")

player_positions = (
    players[["nfl_id", "nfl_position"]]
    .dropna(subset=["nfl_id"])
    .drop_duplicates("nfl_id")
)

Combine profile: rows=107; duplicate keys=0; null-key rows=0
Game player-season profile: rows=146; duplicate keys=0; null-key rows=0
Player IDs with conflicting nfl_position values: 0


## 2. Merge Combine and game player-season profiles

The merge uses both `draft_year` and `nfl_id`; this prevents accidental matching across draft cohorts. Player position is attached from `players.csv`.

In [3]:
merged = game.merge(
    combine,
    on=["draft_year", "nfl_id"],
    how="inner",
    validate="many_to_one",
    suffixes=("_game", "_combine"),
    indicator=True
)

merge_audit = merged["_merge"].value_counts(dropna=False).rename_axis("merge_status").reset_index(name="rows")
merged = merged.drop(columns="_merge")

merged = merged.merge(
    player_positions,
    on="nfl_id",
    how="left",
    validate="many_to_one"
)

print("Merged shape:", merged.shape)
print("\\nMerge status:")
display(merge_audit)
print("\\nPosition distribution:")
display(merged["nfl_position"].value_counts(dropna=False))

Merged shape: (146, 70)
\nMerge status:


,merge_status,rows
0,both,146
1,left_only,0
2,right_only,0


\nPosition distribution:


nfl_position
WR    143
TE      3
Name: count, dtype: int64

## 3. Apply eligibility rules and record sample flow

Rules:
1. Position must be `WR` or `SLOT_WR`.
2. Game season must be greater than or equal to draft year.
3. At least 10 receiver plays are required for the primary analysis.

Every exclusion is counted and reported.

In [4]:
# Coerce the relevant identifiers and exposure variables to numeric safely.
for col in ["draft_year", "season", "receiver_plays"]:
    merged[col] = pd.to_numeric(merged[col], errors="coerce")

sample_flow = []
def record_step(label, before, after):
    sample_flow.append({
        "step": label,
        "rows_before": int(before),
        "rows_after": int(after),
        "rows_removed": int(before - after)
    })

working = merged.copy()
record_step("Merged game seasons with Combine profiles", len(game), len(working))

before = len(working)
working = working[working["nfl_position"].isin(["WR", "SLOT_WR"])].copy()
record_step("Keep WR and SLOT_WR", before, len(working))

before = len(working)
working = working[working["season"] >= working["draft_year"]].copy()
record_step("Keep seasons at or after draft year", before, len(working))

before = len(working)
working = working[working["receiver_plays"] >= 10].copy()
record_step("Primary sample: at least 10 receiver plays", before, len(working))

sample_flow_df = pd.DataFrame(sample_flow)
display(sample_flow_df)

if working.empty:
    raise ValueError("Primary analysis sample is empty. Inspect merge and eligibility rules.")

# Final expected grain: one player-season per row.
key_audit(working, GAME_KEY, "Primary analysis sample")

print("Primary sample rows:", len(working))
print("Unique players:", working["nfl_id"].nunique())
print("Draft cohorts:")
display(working.groupby("draft_year").agg(
    player_seasons=("nfl_id", "size"),
    unique_players=("nfl_id", "nunique"),
    seasons=("season", "nunique")
).reset_index())

,step,rows_before,rows_after,rows_removed
0,Merged game seasons with Combine profiles,146,146,0
1,Keep WR and SLOT_WR,146,143,3
2,Keep seasons at or after draft year,143,143,0
3,Primary sample: at least 10 receiver plays,143,114,29


Primary analysis sample: rows=114; duplicate keys=0; null-key rows=0
Primary sample rows: 114
Unique players: 65
Draft cohorts:


,draft_year,player_seasons,unique_players,seasons
0,2023,62,27,3
1,2024,35,21,2
2,2025,17,17,1


In [6]:
print("=" * 90)
print("GAME PLAYER-SEASON PROFILE — ACTUAL COLUMNS")
print("=" * 90)

for col in game.columns:
    print(col)

print("\n" + "=" * 90)
print("MERGED DATASET — MOVEMENT-RELATED COLUMNS")
print("=" * 90)

for col in working.columns:
    if any(term in col.lower() for term in [
        "speed", "acceleration", "efficiency",
        "displacement", "turn", "yard", "forty",
        "vertical", "jump", "shuttle", "cone"
    ]):
        print(col)

GAME PLAYER-SEASON PROFILE — ACTUAL COLUMNS
draft_year
season
nfl_id
game_mean_speed_raw_median
game_mean_speed_raw_mean
game_peak_speed_raw_median
game_peak_speed_raw_mean
game_p95_speed_raw_median
game_p95_speed_raw_mean
game_mean_acceleration_raw_median
game_mean_acceleration_raw_mean
game_peak_acceleration_raw_median
game_peak_acceleration_raw_mean
game_minimum_acceleration_raw_median
game_minimum_acceleration_raw_mean
game_summed_recorded_displacement_raw_median
game_summed_recorded_displacement_raw_mean
game_coordinate_path_length_raw_median
game_coordinate_path_length_raw_mean
game_total_direction_change_raw_median
game_total_direction_change_raw_mean
game_path_efficiency_median
game_path_efficiency_mean
game_snap_to_throw_seconds_median
game_snap_to_throw_seconds_mean
game_separation_at_pass_forward_median
game_separation_at_pass_forward_mean
game_receiving_yards_median
game_receiving_yards_mean
game_tracking_observations_median
game_maximum_sampling_gap_seconds_median
game_max

In [8]:

# ============================================================
# 4. DEFINE RESEARCH OUTCOMES AND PREDICTOR GROUPS
# ============================================================

# Load traditional Combine test results separately.
COMBINE_RESULTS_PATH = DATA_RAW / "combine_results.csv"

if not COMBINE_RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Traditional Combine results not found: {COMBINE_RESULTS_PATH}"
    )

combine_results = pd.read_csv(
    COMBINE_RESULTS_PATH,
    low_memory=False
)

# Validate join keys.
require_columns(
    combine_results,
    ["draft_year", "nfl_id"],
    "combine_results.csv"
)

combine_results["draft_year"] = pd.to_numeric(
    combine_results["draft_year"], errors="coerce"
)
combine_results["nfl_id"] = pd.to_numeric(
    combine_results["nfl_id"], errors="coerce"
)

# Check traditional Combine results grain.
key_audit(
    combine_results,
    ["draft_year", "nfl_id"],
    "Traditional Combine results"
)

# Attach traditional test results to the existing analysis population.
working = working.merge(
    combine_results,
    on=["draft_year", "nfl_id"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_results")
)

print("Analysis table after traditional Combine join:", working.shape)

# ------------------------------------------------------------
# A. SIX REGULAR-SEASON GAME MOVEMENT OUTCOMES
# ------------------------------------------------------------

OUTCOMES = [
    "game_mean_speed_raw_median",
    "game_peak_speed_raw_median",
    "game_p95_speed_raw_median",
    "game_mean_acceleration_raw_median",
    "game_peak_acceleration_raw_median",
    "game_path_efficiency_median",
]

# ------------------------------------------------------------
# B. SIX TRADITIONAL COMBINE PREDICTORS
# ------------------------------------------------------------

TRADITIONAL_PREDICTORS = [
    "forty",
    "vertical",
    "broad_jump",
    "three_cone",
    "short_shuttle",
    "ten_yd_split",
]

# ------------------------------------------------------------
# C. SIX COMBINE TRACKING-DERIVED PREDICTORS
# ------------------------------------------------------------

TRACKING_PREDICTORS = [
    "combine_mean_speed_median_across_drills",
    "combine_peak_speed_median_across_drills",
    "combine_p95_speed_median_across_drills",
    "combine_mean_acceleration_median_across_drills",
    "combine_peak_acceleration_median_across_drills",
    "combine_path_efficiency_median_across_drills",
]

# ------------------------------------------------------------
# D. VALIDATE ALL REQUIRED VARIABLES
# ------------------------------------------------------------

required_features = (
    OUTCOMES
    + TRADITIONAL_PREDICTORS
    + TRACKING_PREDICTORS
)

missing_features = [
    col for col in required_features
    if col not in working.columns
]

if missing_features:
    raise KeyError(
        "The following research variables are missing:\n"
        + "\n".join(missing_features)
    )

# ------------------------------------------------------------
# E. CONVERT MODELLING VARIABLES TO NUMERIC
# ------------------------------------------------------------

for col in required_features:
    working[col] = pd.to_numeric(
        working[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# F. PRINT RESEARCH VARIABLE DEFINITIONS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("RESEARCH VARIABLE DEFINITION AUDIT")
print("=" * 90)

print("\nGAME MOVEMENT OUTCOMES:")
for col in OUTCOMES:
    print("  ", col)

print("\nTRADITIONAL COMBINE PREDICTORS:")
for col in TRADITIONAL_PREDICTORS:
    print("  ", col)

print("\nCOMBINE TRACKING PREDICTORS:")
for col in TRACKING_PREDICTORS:
    print("  ", col)

print("\nVARIABLE COUNTS:")
print("  Outcomes:", len(OUTCOMES))
print("  Traditional predictors:", len(TRADITIONAL_PREDICTORS))
print("  Tracking predictors:", len(TRACKING_PREDICTORS))
print("  Total:", len(required_features))

assert len(OUTCOMES) == 6
assert len(TRADITIONAL_PREDICTORS) == 6
assert len(TRACKING_PREDICTORS) == 6
assert len(required_features) == 18

print("\nVARIABLE DEFINITION STATUS: PASS")


Traditional Combine results: rows=510; duplicate keys=0; null-key rows=0
Analysis table after traditional Combine join: (114, 86)

RESEARCH VARIABLE DEFINITION AUDIT

GAME MOVEMENT OUTCOMES:
   game_mean_speed_raw_median
   game_peak_speed_raw_median
   game_p95_speed_raw_median
   game_mean_acceleration_raw_median
   game_peak_acceleration_raw_median
   game_path_efficiency_median

TRADITIONAL COMBINE PREDICTORS:
   forty
   vertical
   broad_jump
   three_cone
   short_shuttle
   ten_yd_split

COMBINE TRACKING PREDICTORS:
   combine_mean_speed_median_across_drills
   combine_peak_speed_median_across_drills
   combine_p95_speed_median_across_drills
   combine_mean_acceleration_median_across_drills
   combine_peak_acceleration_median_across_drills
   combine_path_efficiency_median_across_drills

VARIABLE COUNTS:
  Outcomes: 6
  Traditional predictors: 6
  Tracking predictors: 6
  Total: 18

VARIABLE DEFINITION STATUS: PASS


## 4. Define research outcomes and predictor groups

The variable lists below reflect the current research design. The notebook validates that every requested variable exists before exporting. If a source column has a different name, inspect the printed schema and update the mapping explicitly—do not silently substitute a different measurement.

In [9]:

# ============================================================
# 4. DEFINE RESEARCH OUTCOMES AND PREDICTOR GROUPS
# ============================================================

# Load traditional Combine results
COMBINE_RESULTS_PATH = DATA_RAW / "combine_results.csv"

if not COMBINE_RESULTS_PATH.exists():
    raise FileNotFoundError(COMBINE_RESULTS_PATH)

combine_results = pd.read_csv(
    COMBINE_RESULTS_PATH,
    low_memory=False
)

combine_results["draft_year"] = pd.to_numeric(
    combine_results["draft_year"], errors="coerce"
)

combine_results["nfl_id"] = pd.to_numeric(
    combine_results["nfl_id"], errors="coerce"
)

key_audit(
    combine_results,
    ["draft_year", "nfl_id"],
    "Traditional Combine results"
)

# Attach traditional Combine tests
working = working.merge(
    combine_results,
    on=["draft_year", "nfl_id"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_results")
)

# SIX GAME MOVEMENT OUTCOMES
OUTCOMES = [
    "game_mean_speed_raw_median",
    "game_peak_speed_raw_median",
    "game_p95_speed_raw_median",
    "game_mean_acceleration_raw_median",
    "game_peak_acceleration_raw_median",
    "game_path_efficiency_median",
]

# SIX TRADITIONAL COMBINE PREDICTORS
TRADITIONAL_PREDICTORS = [
    "forty",
    "vertical",
    "broad_jump",
    "three_cone",
    "short_shuttle",
    "ten_yd_split",
]

# SIX COMBINE TRACKING PREDICTORS
TRACKING_PREDICTORS = [
    "combine_mean_speed_median_across_drills",
    "combine_peak_speed_median_across_drills",
    "combine_p95_speed_median_across_drills",
    "combine_mean_acceleration_median_across_drills",
    "combine_peak_acceleration_median_across_drills",
    "combine_path_efficiency_median_across_drills",
]

# Validate all variables
required_features = (
    OUTCOMES
    + TRADITIONAL_PREDICTORS
    + TRACKING_PREDICTORS
)

missing = [
    col for col in required_features
    if col not in working.columns
]

if missing:
    raise KeyError(f"Missing research variables: {missing}")

# Numeric conversion
for col in required_features:
    working[col] = pd.to_numeric(
        working[col],
        errors="coerce"
    )

print("=" * 90)
print("RESEARCH VARIABLE DEFINITION AUDIT")
print("=" * 90)

print("\nOUTCOMES:")
print(*OUTCOMES, sep="\n")

print("\nTRADITIONAL PREDICTORS:")
print(*TRADITIONAL_PREDICTORS, sep="\n")

print("\nTRACKING PREDICTORS:")
print(*TRACKING_PREDICTORS, sep="\n")

print("\nVARIABLE COUNTS")
print("Outcomes:", len(OUTCOMES))
print("Traditional predictors:", len(TRADITIONAL_PREDICTORS))
print("Tracking predictors:", len(TRACKING_PREDICTORS))
print("Total:", len(required_features))

assert len(required_features) == 18

print("\nSECTION 4 STATUS: PASS")


Traditional Combine results: rows=510; duplicate keys=0; null-key rows=0
RESEARCH VARIABLE DEFINITION AUDIT

OUTCOMES:
game_mean_speed_raw_median
game_peak_speed_raw_median
game_p95_speed_raw_median
game_mean_acceleration_raw_median
game_peak_acceleration_raw_median
game_path_efficiency_median

TRADITIONAL PREDICTORS:
forty
vertical
broad_jump
three_cone
short_shuttle
ten_yd_split

TRACKING PREDICTORS:
combine_mean_speed_median_across_drills
combine_peak_speed_median_across_drills
combine_p95_speed_median_across_drills
combine_mean_acceleration_median_across_drills
combine_peak_acceleration_median_across_drills
combine_path_efficiency_median_across_drills

VARIABLE COUNTS
Outcomes: 6
Traditional predictors: 6
Tracking predictors: 6
Total: 18

SECTION 4 STATUS: PASS


## 5. Numeric conversion and feature-coverage audit

We preserve the original columns and create a modelling-ready numeric copy of the selected variables. Missing values are not imputed in this notebook.

In [12]:

# ============================================================
# NOTEBOOK 06 — SECTION 5
# FEATURE COVERAGE AND COMPLETE-CASE AUDIT
# ============================================================

selected_features = (
    OUTCOMES
    + TRADITIONAL_PREDICTORS
    + TRACKING_PREDICTORS
)

analysis = working.copy()

for col in selected_features:
    analysis[col] = pd.to_numeric(
        analysis[col], errors="coerce"
    )

# Feature-level coverage
coverage_rows = []

for group_name, cols in [
    ("outcomes", OUTCOMES),
    ("traditional_predictors", TRADITIONAL_PREDICTORS),
    ("tracking_predictors", TRACKING_PREDICTORS)
]:
    for col in cols:
        coverage_rows.append({
            "group": group_name,
            "feature": col,
            "non_missing": analysis[col].notna().sum(),
            "missing": analysis[col].isna().sum(),
            "coverage_pct": round(
                analysis[col].notna().mean() * 100, 2
            )
        })

coverage = pd.DataFrame(coverage_rows)

print("FEATURE-LEVEL COVERAGE")
display(coverage)

# Group-level coverage
group_coverage = []

for group_name, cols in [
    ("outcomes", OUTCOMES),
    ("traditional_predictors", TRADITIONAL_PREDICTORS),
    ("tracking_predictors", TRACKING_PREDICTORS)
]:
    mask = analysis[cols].notna().all(axis=1)

    group_coverage.append({
        "group": group_name,
        "features": len(cols),
        "complete_player_seasons": int(mask.sum()),
        "incomplete_player_seasons": int((~mask).sum()),
        "complete_pct": round(mask.mean() * 100, 2)
    })

group_coverage_df = pd.DataFrame(group_coverage)

print("\nGROUP-LEVEL COVERAGE")
display(group_coverage_df)

# Common complete-case population
COMMON_FEATURES = selected_features

analysis["complete_all_model_features"] = (
    analysis[COMMON_FEATURES].notna().all(axis=1)
)

common_complete = analysis.loc[
    analysis["complete_all_model_features"]
].copy()

print("\nCOMMON COMPLETE-CASE SAMPLE")
print("Primary sample:", len(analysis))
print("Complete cases:", len(common_complete))
print("Excluded:", len(analysis) - len(common_complete))

if len(analysis) > 0:
    print(
        "Complete-case percentage:",
        round(len(common_complete) / len(analysis) * 100, 2),
        "%"
    )

print("\nSECTION 5 STATUS: PASS")


FEATURE-LEVEL COVERAGE


,group,feature,non_missing,missing,coverage_pct
0,outcomes,game_mean_speed_raw_median,114,0,100.00
1,outcomes,game_peak_speed_raw_median,114,0,100.00
2,outcomes,game_p95_speed_raw_median,114,0,100.00
3,outcomes,game_mean_acceleration_raw_median,114,0,100.00
4,outcomes,game_peak_acceleration_raw_median,114,0,100.00
5,outcomes,game_path_efficiency_median,114,0,100.00
6,traditional_predictors,forty,102,12,89.47
7,traditional_predictors,vertical,100,14,87.72
8,traditional_predictors,broad_jump,102,12,89.47
9,traditional_predictors,three_cone,35,79,30.70



GROUP-LEVEL COVERAGE


,group,features,complete_player_seasons,incomplete_player_seasons,complete_pct
0,outcomes,6,114,0,100.00
1,traditional_predictors,6,26,88,22.81
2,tracking_predictors,6,114,0,100.00



COMMON COMPLETE-CASE SAMPLE
Primary sample: 114
Complete cases: 26
Excluded: 88
Complete-case percentage: 22.81 %

SECTION 5 STATUS: PASS


## 6. Final integrity checks and exports

Exports include the full eligible primary sample, a common complete-case sample, sample flow, feature coverage, and a machine-readable manifest.

In [14]:

# ============================================================
# NOTEBOOK 06 — SECTION 6
# FINAL INTEGRITY CHECKS AND DATASET EXPORTS
# ============================================================

print("=" * 90)
print("FINAL ANALYSIS DATASET INTEGRITY AUDIT")
print("=" * 90)

# ------------------------------------------------------------
# 1. VALIDATE PRIMARY DATASET
# ------------------------------------------------------------

GAME_KEY = ["draft_year", "nfl_id", "season"]

key_audit(
    analysis,
    GAME_KEY,
    "Final primary analysis dataset"
)

assert (analysis["season"] >= analysis["draft_year"]).all()
assert (analysis["receiver_plays"] >= 10).all()

assert analysis["nfl_position"].isin(
    ["WR", "SLOT_WR"]
).all()

# ------------------------------------------------------------
# 2. REBUILD COMMON COMPLETE-CASE DATASET
# ------------------------------------------------------------

COMMON_FEATURES = (
    OUTCOMES
    + TRADITIONAL_PREDICTORS
    + TRACKING_PREDICTORS
)

analysis["complete_all_model_features"] = (
    analysis[COMMON_FEATURES].notna().all(axis=1)
)

common_complete = analysis.loc[
    analysis["complete_all_model_features"]
].copy()

# ------------------------------------------------------------
# 3. VALIDATE SAMPLE COUNTS
# ------------------------------------------------------------

print("\nPRIMARY SAMPLE")
print("Player-seasons:", len(analysis))
print("Unique players:", analysis["nfl_id"].nunique())

print("\nCOMMON COMPLETE-CASE SAMPLE")
print("Player-seasons:", len(common_complete))
print("Unique players:", common_complete["nfl_id"].nunique())

print(
    "Excluded player-seasons:",
    len(analysis) - len(common_complete)
)

# ------------------------------------------------------------
# 4. CREATE OUTPUT DIRECTORIES
# ------------------------------------------------------------

DATA_FEATURES.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 5. EXPORT PARQUET DATASETS
# ------------------------------------------------------------

primary_path = (
    DATA_FEATURES / "wr_analysis_dataset_primary.parquet"
)

common_path = (
    DATA_FEATURES / "wr_analysis_dataset_common_complete.parquet"
)

analysis.to_parquet(
    primary_path,
    index=False
)

common_complete.to_parquet(
    common_path,
    index=False
)

# ------------------------------------------------------------
# 6. EXPORT AUDIT REPORTS
# ------------------------------------------------------------

sample_flow_df.to_csv(
    REPORTS / "sample_flow.csv",
    index=False
)

coverage.to_csv(
    REPORTS / "feature_coverage.csv",
    index=False
)

group_coverage_df.to_csv(
    REPORTS / "feature_group_coverage.csv",
    index=False
)

# ------------------------------------------------------------
# 7. CREATE REPRODUCIBILITY MANIFEST
# ------------------------------------------------------------

manifest = {
    "notebook": "06_analysis_dataset_construction",

    "primary_sample_rows": int(len(analysis)),
    "primary_unique_players": int(
        analysis["nfl_id"].nunique()
    ),

    "common_complete_rows": int(
        len(common_complete)
    ),

    "common_complete_unique_players": int(
        common_complete["nfl_id"].nunique()
    ),

    "primary_key": GAME_KEY,

    "outcomes": OUTCOMES,

    "traditional_predictors": TRADITIONAL_PREDICTORS,

    "tracking_predictors": TRACKING_PREDICTORS,

    "minimum_receiver_plays": 10,

    "eligible_positions": [
        "WR",
        "SLOT_WR"
    ],

    "imputation_performed": False,

    "source_tables": [
        "wr_combine_player_profiles.parquet",
        "wr_game_player_season_profiles.parquet",
        "combine_results.csv"
    ]
}

manifest_path = (
    REPORTS / "analysis_dataset_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 8. FINAL OUTPUT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("EXPORTED FILES")
print("=" * 90)

for path in [
    primary_path,
    common_path,
    REPORTS / "sample_flow.csv",
    REPORTS / "feature_coverage.csv",
    REPORTS / "feature_group_coverage.csv",
    manifest_path
]:
    print(
        f"{path} | {path.stat().st_size:,} bytes"
    )

print("\n" + "=" * 90)
print("FINAL SAMPLE BY DRAFT YEAR AND SEASON")
print("=" * 90)

final_cohort_summary = (
    analysis
    .groupby(["draft_year", "season"])
    .agg(
        player_seasons=("nfl_id", "size"),
        unique_players=("nfl_id", "nunique"),
        median_receiver_plays=("receiver_plays", "median")
    )
    .reset_index()
)

display(final_cohort_summary)

print("\n" + "=" * 90)
print("NOTEBOOK 06 EXPORT STATUS: PASS")
print("=" * 90)


FINAL ANALYSIS DATASET INTEGRITY AUDIT
Final primary analysis dataset: rows=114; duplicate keys=0; null-key rows=0

PRIMARY SAMPLE
Player-seasons: 114
Unique players: 65

COMMON COMPLETE-CASE SAMPLE
Player-seasons: 26
Unique players: 16
Excluded player-seasons: 88

EXPORTED FILES
D:\PROJECTS\WR-MovementIQ\data\features\wr_analysis_dataset_primary.parquet | 134,168 bytes
D:\PROJECTS\WR-MovementIQ\data\features\wr_analysis_dataset_common_complete.parquet | 98,723 bytes
D:\PROJECTS\WR-MovementIQ\reports\analysis_dataset\sample_flow.csv | 228 bytes
D:\PROJECTS\WR-MovementIQ\reports\analysis_dataset\feature_coverage.csv | 1,095 bytes
D:\PROJECTS\WR-MovementIQ\reports\analysis_dataset\feature_group_coverage.csv | 176 bytes
D:\PROJECTS\WR-MovementIQ\reports\analysis_dataset\analysis_dataset_manifest.json | 1,268 bytes

FINAL SAMPLE BY DRAFT YEAR AND SEASON


,draft_year,season,player_seasons,unique_players,median_receiver_plays
0,2023,2023,23,23,44.0
1,2023,2024,23,23,46.0
2,2023,2025,16,16,56.0
3,2024,2024,18,18,49.0
4,2024,2025,17,17,52.0
5,2025,2025,17,17,31.0



NOTEBOOK 06 EXPORT STATUS: PASS


In [15]:

print("=" * 90)
print("SAMPLE RECONCILIATION AUDIT")
print("=" * 90)

print("\n1. SOURCE COUNTS")
print("Game player-season profiles:", len(game))
print("Unique game-profile players:", game["nfl_id"].nunique())

print("Combine player profiles:", len(combine))
print("Unique Combine-profile players:", combine["nfl_id"].nunique())

print("Merged before eligibility:", len(merged))

print("\n2. SAMPLE FLOW")
display(sample_flow_df)

print("\n3. GAME PLAYERS WITHOUT COMBINE PROFILES")

game_player_keys = game[["draft_year", "nfl_id"]].drop_duplicates()

combine_player_keys = combine[
    ["draft_year", "nfl_id"]
].drop_duplicates()

missing_combine = game_player_keys.merge(
    combine_player_keys,
    on=["draft_year", "nfl_id"],
    how="left",
    indicator=True
)

missing_combine = missing_combine[
    missing_combine["_merge"] == "left_only"
]

print("Game-profile player keys without Combine profile:",
      len(missing_combine))

display(missing_combine.head(30))

print("\n4. FEATURE GROUP COVERAGE")
display(group_coverage_df)

print("\n5. INDIVIDUAL FEATURE COVERAGE")
display(
    coverage.sort_values(
        ["group", "coverage_pct"],
        ascending=[True, True]
    )
)

print("\n6. COMPLETE-CASE COUNTS BY DRAFT YEAR")

display(
    analysis.groupby("draft_year").agg(
        primary_rows=("nfl_id", "size"),
        primary_players=("nfl_id", "nunique"),
        complete_rows=("complete_all_model_features", "sum")
    ).reset_index()
)


SAMPLE RECONCILIATION AUDIT

1. SOURCE COUNTS
Game player-season profiles: 146
Unique game-profile players: 79
Combine player profiles: 107
Unique Combine-profile players: 107
Merged before eligibility: 146

2. SAMPLE FLOW


,step,rows_before,rows_after,rows_removed
0,Merged game seasons with Combine profiles,146,146,0
1,Keep WR and SLOT_WR,146,143,3
2,Keep seasons at or after draft year,143,143,0
3,Primary sample: at least 10 receiver plays,143,114,29



3. GAME PLAYERS WITHOUT COMBINE PROFILES
Game-profile player keys without Combine profile: 0


,draft_year,nfl_id,_merge



4. FEATURE GROUP COVERAGE


,group,features,complete_player_seasons,incomplete_player_seasons,complete_pct
0,outcomes,6,114,0,100.00
1,traditional_predictors,6,26,88,22.81
2,tracking_predictors,6,114,0,100.00



5. INDIVIDUAL FEATURE COVERAGE


,group,feature,non_missing,missing,coverage_pct
0,outcomes,game_mean_speed_raw_median,114,0,100.00
1,outcomes,game_peak_speed_raw_median,114,0,100.00
2,outcomes,game_p95_speed_raw_median,114,0,100.00
3,outcomes,game_mean_acceleration_raw_median,114,0,100.00
4,outcomes,game_peak_acceleration_raw_median,114,0,100.00
5,outcomes,game_path_efficiency_median,114,0,100.00
12,tracking_predictors,combine_mean_speed_median_across_drills,114,0,100.00
13,tracking_predictors,combine_peak_speed_median_across_drills,114,0,100.00
14,tracking_predictors,combine_p95_speed_median_across_drills,114,0,100.00
15,tracking_predictors,combine_mean_acceleration_median_across_drills,114,0,100.00



6. COMPLETE-CASE COUNTS BY DRAFT YEAR


,draft_year,primary_rows,primary_players,complete_rows
0,2023,62,27,10
1,2024,35,21,10
2,2025,17,17,6


## 7. Interpretation and handoff to Notebook 07

Before proceeding, verify:
- The primary sample size and player count are plausible relative to the earlier reference results.
- The expected six outcomes and 12 predictors resolved without substitutions.
- The common complete-case sample is large enough for player-grouped validation.
- Missingness is not concentrated in one draft cohort in a way that changes the research population.

**Do not impute missing values or select models here.** Notebook 07 should perform the modelling-sample audit and establish the exact player-grouped validation folds.

In [16]:

import pandas as pd

print("=" * 90)
print("WR-MOVEMENTIQ — SAMPLE RECONCILIATION")
print("=" * 90)

# 1. Source population counts
print("\n1. SOURCE POPULATIONS")

for name, df in [
    ("Game profiles", game),
    ("Combine profiles", combine),
    ("Merged profiles", merged),
    ("Final analysis", analysis),
]:
    print(
        f"{name:25s} rows={len(df):5,d} | "
        f"players={df['nfl_id'].nunique():4,d}"
    )

# 2. Identify game player keys without Combine profiles
print("\n2. GAME PROFILES WITHOUT MATCHING COMBINE PROFILES")

game_keys = game[["draft_year", "nfl_id"]].drop_duplicates()
combine_keys = combine[["draft_year", "nfl_id"]].drop_duplicates()

key_comparison = game_keys.merge(
    combine_keys,
    on=["draft_year", "nfl_id"],
    how="left",
    indicator=True
)

missing_keys = key_comparison[
    key_comparison["_merge"] == "left_only"
].drop(columns="_merge")

print("Unmatched game player keys:", len(missing_keys))
display(missing_keys.head(30))

# 3. Reconcile primary sample by draft year
print("\n3. PRIMARY SAMPLE BY DRAFT YEAR")

display(
    analysis.groupby("draft_year").agg(
        player_seasons=("nfl_id", "size"),
        unique_players=("nfl_id", "nunique"),
        median_receiver_plays=("receiver_plays", "median")
    ).reset_index()
)

# 4. Missingness by individual feature
print("\n4. MISSINGNESS BY FEATURE")

feature_list = (
    OUTCOMES
    + TRADITIONAL_PREDICTORS
    + TRACKING_PREDICTORS
)

missingness = pd.DataFrame({
    "feature": feature_list,
    "missing_count": [
        analysis[c].isna().sum() for c in feature_list
    ],
    "available_count": [
        analysis[c].notna().sum() for c in feature_list
    ],
    "coverage_pct": [
        analysis[c].notna().mean() * 100 for c in feature_list
    ]
})

display(
    missingness.sort_values("coverage_pct")
    .round(2)
    .reset_index(drop=True)
)

# 5. Compare model-specific sample sizes
print("\n5. MODEL-SPECIFIC COMPLETE CASES")

outcome_complete = analysis[OUTCOMES].notna().all(axis=1)
traditional_complete = analysis[
    OUTCOMES + TRADITIONAL_PREDICTORS
].notna().all(axis=1)

tracking_complete = analysis[
    OUTCOMES + TRACKING_PREDICTORS
].notna().all(axis=1)

combined_complete = analysis[
    OUTCOMES + TRADITIONAL_PREDICTORS + TRACKING_PREDICTORS
].notna().all(axis=1)

model_samples = []

for label, mask in [
    ("Outcomes only", outcome_complete),
    ("Traditional model", traditional_complete),
    ("Tracking model", tracking_complete),
    ("Combined model", combined_complete),
]:
    subset = analysis.loc[mask]
    model_samples.append({
        "sample": label,
        "player_seasons": len(subset),
        "unique_players": subset["nfl_id"].nunique()
    })

display(pd.DataFrame(model_samples))

# 6. Missingness by predictor group
print("\n6. PREDICTOR GROUP COVERAGE")

for label, columns in [
    ("Outcomes", OUTCOMES),
    ("Traditional", TRADITIONAL_PREDICTORS),
    ("Tracking", TRACKING_PREDICTORS),
]:
    print(
        f"{label:15s}: "
        f"{analysis[columns].notna().all(axis=1).sum():3d} "
        f"complete rows out of {len(analysis)}"
    )

# 7. Compare with previously expected sample
print("\n7. SAMPLE SIZE RECONCILIATION")
print("Current primary rows:", len(analysis))
print("Previously reported target:", 116)
print("Difference:", len(analysis) - 116)
print("Current unique players:", analysis["nfl_id"].nunique())
print("Previously reported players:", 66)

# 8. Existing sample-flow report
print("\n8. SAMPLE FLOW")
display(sample_flow_df)

print("\n" + "=" * 90)
print("RECONCILIATION AUDIT COMPLETE")
print("=" * 90)


WR-MOVEMENTIQ — SAMPLE RECONCILIATION

1. SOURCE POPULATIONS
Game profiles             rows=  146 | players=  79
Combine profiles          rows=  107 | players= 107
Merged profiles           rows=  146 | players=  79
Final analysis            rows=  114 | players=  65

2. GAME PROFILES WITHOUT MATCHING COMBINE PROFILES
Unmatched game player keys: 0


,draft_year,nfl_id



3. PRIMARY SAMPLE BY DRAFT YEAR


,draft_year,player_seasons,unique_players,median_receiver_plays
0,2023,62,27,50.5
1,2024,35,21,52.0
2,2025,17,17,31.0



4. MISSINGNESS BY FEATURE


,feature,missing_count,available_count,coverage_pct
0,three_cone,79,35,30.70
1,short_shuttle,68,46,40.35
2,vertical,14,100,87.72
3,broad_jump,12,102,89.47
4,ten_yd_split,12,102,89.47
5,forty,12,102,89.47
6,combine_mean_acceleration_median_across_drills,0,114,100.00
7,combine_p95_speed_median_across_drills,0,114,100.00
8,combine_peak_speed_median_across_drills,0,114,100.00
9,combine_mean_speed_median_across_drills,0,114,100.00



5. MODEL-SPECIFIC COMPLETE CASES


,sample,player_seasons,unique_players
0,Outcomes only,114,65
1,Traditional model,26,16
2,Tracking model,114,65
3,Combined model,26,16



6. PREDICTOR GROUP COVERAGE
Outcomes       : 114 complete rows out of 114
Traditional    :  26 complete rows out of 114
Tracking       : 114 complete rows out of 114

7. SAMPLE SIZE RECONCILIATION
Current primary rows: 114
Previously reported target: 116
Difference: -2
Current unique players: 65
Previously reported players: 66

8. SAMPLE FLOW


,step,rows_before,rows_after,rows_removed
0,Merged game seasons with Combine profiles,146,146,0
1,Keep WR and SLOT_WR,146,143,3
2,Keep seasons at or after draft year,143,143,0
3,Primary sample: at least 10 receiver plays,143,114,29



RECONCILIATION AUDIT COMPLETE


In [17]:

# ============================================================
# SECTION 9 — FAIR MODEL COMPARISON SAMPLE CONSTRUCTION
# ============================================================

CORE_TRADITIONAL = [
    "forty",
    "vertical",
    "broad_jump"
]

EXTENDED_TRADITIONAL = [
    "forty",
    "vertical",
    "broad_jump",
    "ten_yd_split",
    "three_cone",
    "short_shuttle"
]

TRACKING = TRACKING_PREDICTORS.copy()

print("=" * 90)
print("FAIR MODEL COMPARISON SAMPLE AUDIT")
print("=" * 90)

# Evaluate each outcome separately.
sample_results = []

for outcome in OUTCOMES:

    specifications = {
        "Baseline": [],
        "Traditional_Core": CORE_TRADITIONAL,
        "Traditional_Extended": EXTENDED_TRADITIONAL,
        "Tracking_Only": TRACKING,
        "Combined_Core": CORE_TRADITIONAL + TRACKING,
        "Combined_Extended": EXTENDED_TRADITIONAL + TRACKING,
    }

    for model_name, predictors in specifications.items():

        required = [outcome] + predictors
        mask = analysis[required].notna().all(axis=1)
        subset = analysis.loc[mask]

        sample_results.append({
            "outcome": outcome,
            "model": model_name,
            "player_seasons": len(subset),
            "unique_players": subset["nfl_id"].nunique(),
            "draft_years": subset["draft_year"].nunique()
        })

sample_results_df = pd.DataFrame(sample_results)

display(
    sample_results_df.sort_values(
        ["outcome", "model"]
    ).reset_index(drop=True)
)

# Overall common sample for the six outcomes and core predictors.
CORE_FEATURES = (
    OUTCOMES
    + CORE_TRADITIONAL
    + TRACKING
)

core_mask = analysis[CORE_FEATURES].notna().all(axis=1)
core_common = analysis.loc[core_mask].copy()

print("\nCORE COMMON SAMPLE")
print("Player-seasons:", len(core_common))
print("Unique players:", core_common["nfl_id"].nunique())

print("\nCORE SAMPLE BY DRAFT YEAR")
display(
    core_common.groupby("draft_year").agg(
        player_seasons=("nfl_id", "size"),
        unique_players=("nfl_id", "nunique")
    ).reset_index()
)

# Identify exactly which observations are excluded from the
# core combined model and why.
excluded = analysis.loc[~core_mask].copy()

print("\nOBSERVATIONS EXCLUDED FROM CORE COMBINED SAMPLE")
print("Excluded player-seasons:", len(excluded))
print("Excluded unique players:", excluded["nfl_id"].nunique())

display(
    excluded[
        ["draft_year", "nfl_id", "season", "receiver_plays"]
        + CORE_TRADITIONAL
    ].sort_values(["draft_year", "nfl_id"])
)

# Export results.
REPORT_DIR = PROJECT_ROOT / "reports" / "analysis_dataset"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

sample_results_df.to_csv(
    REPORT_DIR / "model_specific_sample_sizes.csv",
    index=False
)

core_common.to_parquet(
    DATA_FEATURES / "wr_analysis_dataset_core_common.parquet",
    index=False
)

print("\nExported:")
print(REPORT_DIR / "model_specific_sample_sizes.csv")
print(DATA_FEATURES / "wr_analysis_dataset_core_common.parquet")

print("\nSECTION 9 COMPLETE")


FAIR MODEL COMPARISON SAMPLE AUDIT


,outcome,model,player_seasons,unique_players,draft_years
0,game_mean_acceleration_raw_median,Baseline,114,65,3
1,game_mean_acceleration_raw_median,Combined_Core,93,54,3
2,game_mean_acceleration_raw_median,Combined_Extended,26,16,3
3,game_mean_acceleration_raw_median,Tracking_Only,114,65,3
4,game_mean_acceleration_raw_median,Traditional_Core,93,54,3
5,game_mean_acceleration_raw_median,Traditional_Extended,26,16,3
6,game_mean_speed_raw_median,Baseline,114,65,3
7,game_mean_speed_raw_median,Combined_Core,93,54,3
8,game_mean_speed_raw_median,Combined_Extended,26,16,3
9,game_mean_speed_raw_median,Tracking_Only,114,65,3



CORE COMMON SAMPLE
Player-seasons: 93
Unique players: 54

CORE SAMPLE BY DRAFT YEAR


,draft_year,player_seasons,unique_players
0,2023,45,20
1,2024,34,20
2,2025,14,14



OBSERVATIONS EXCLUDED FROM CORE COMBINED SAMPLE
Excluded player-seasons: 21
Excluded unique players: 11


,draft_year,nfl_id,season,receiver_plays,forty,vertical,broad_jump
0,2023,55884,2023,90,NaN,35.0,125.0
23,2023,55884,2024,128,NaN,35.0,125.0
46,2023,55884,2025,156,NaN,35.0,125.0
1,2023,55885,2023,66,NaN,40.5,134.0
24,2023,55885,2024,91,NaN,40.5,134.0
47,2023,55885,2025,84,NaN,40.5,134.0
7,2023,55934,2023,74,4.49,NaN,121.0
30,2023,55934,2024,80,4.49,NaN,121.0
12,2023,55990,2023,13,4.36,NaN,NaN
35,2023,55990,2024,13,4.36,NaN,NaN



Exported:
D:\PROJECTS\WR-MovementIQ\reports\analysis_dataset\model_specific_sample_sizes.csv
D:\PROJECTS\WR-MovementIQ\data\features\wr_analysis_dataset_core_common.parquet

SECTION 9 COMPLETE


In [18]:

# ============================================================
# SECTION 10 — RECONCILE THE 114 VS 116 SAMPLE DISCREPANCY
# ============================================================

print("=" * 90)
print("NOTEBOOK 34 VS NOTEBOOK 06 SAMPLE RECONCILIATION")
print("=" * 90)

# Reconstruct the eligible population before the 10-play filter.
reconciliation = merged.copy()

reconciliation["receiver_plays"] = pd.to_numeric(
    reconciliation["receiver_plays"], errors="coerce"
)

# Position eligibility
eligible = reconciliation[
    reconciliation["nfl_position"].isin(["WR", "SLOT_WR"])
].copy()

# Post-draft seasons
eligible = eligible[
    eligible["season"] >= eligible["draft_year"]
].copy()

print("\n1. ELIGIBLE POPULATION BEFORE PLAY FILTER")
print("Rows:", len(eligible))
print("Players:", eligible["nfl_id"].nunique())

# Threshold sensitivity
print("\n2. RECEIVER-PLAY THRESHOLD SENSITIVITY")

threshold_results = []

for threshold in [5, 8, 9, 10, 11, 12, 15]:
    subset = eligible[
        eligible["receiver_plays"] >= threshold
    ]

    threshold_results.append({
        "minimum_receiver_plays": threshold,
        "player_seasons": len(subset),
        "unique_players": subset["nfl_id"].nunique()
    })

display(pd.DataFrame(threshold_results))

# Inspect records excluded by the current 10-play rule.
print("\n3. RECORDS EXCLUDED BY THE 10-PLAY RULE")

below_threshold = eligible[
    eligible["receiver_plays"].isna()
    | (eligible["receiver_plays"] < 10)
].copy()

print("Excluded rows:", len(below_threshold))
print("Excluded players:", below_threshold["nfl_id"].nunique())

display(
    below_threshold[
        [
            "draft_year",
            "nfl_id",
            "season",
            "nfl_position",
            "receiver_plays"
        ]
    ].sort_values(
        ["draft_year", "nfl_id", "season"]
    )
)

# Inspect records that have exactly 9 or 10 plays.
print("\n4. OBSERVATIONS NEAR THE THRESHOLD")

display(
    eligible[
        eligible["receiver_plays"].between(8, 11)
    ][
        [
            "draft_year",
            "nfl_id",
            "season",
            "nfl_position",
            "receiver_plays"
        ]
    ].sort_values(
        ["draft_year", "nfl_id", "season"]
    )
)

# Check repeated player-seasons and unique-player structure.
print("\n5. PLAYER-SEASON KEY AUDIT")

print(
    "Duplicate player-season keys:",
    eligible.duplicated(
        ["draft_year", "nfl_id", "season"]
    ).sum()
)

print(
    "Unique player IDs:",
    eligible["nfl_id"].nunique()
)

print("\n" + "=" * 90)
print("SAMPLE DISCREPANCY DIAGNOSTIC COMPLETE")
print("=" * 90)


NOTEBOOK 34 VS NOTEBOOK 06 SAMPLE RECONCILIATION

1. ELIGIBLE POPULATION BEFORE PLAY FILTER
Rows: 143
Players: 78

2. RECEIVER-PLAY THRESHOLD SENSITIVITY


,minimum_receiver_plays,player_seasons,unique_players
0,5,126,69
1,8,117,67
2,9,114,65
3,10,114,65
4,11,114,65
5,12,113,64
6,15,102,58



3. RECORDS EXCLUDED BY THE 10-PLAY RULE
Excluded rows: 29
Excluded players: 24


,draft_year,nfl_id,season,nfl_position,receiver_plays
59,2023,55904,2025,WR,5
67,2023,55990,2025,WR,2
42,2023,55998,2024,WR,1
15,2023,56052,2023,WR,7
22,2023,56084,2023,WR,2
23,2023,56109,2023,WR,6
74,2023,56109,2025,WR,6
51,2023,56118,2024,WR,6
75,2023,56118,2025,WR,4
52,2023,56121,2024,WR,4



4. OBSERVATIONS NEAR THE THRESHOLD


,draft_year,nfl_id,season,nfl_position,receiver_plays
26,2023,56248,2023,WR,8
28,2023,56541,2023,WR,8
55,2023,56541,2024,WR,11
116,2024,57234,2025,WR,8



5. PLAYER-SEASON KEY AUDIT
Duplicate player-season keys: 0
Unique player IDs: 78

SAMPLE DISCREPANCY DIAGNOSTIC COMPLETE
